# 01 · Exploration et contrôle qualité

Objectif : connaître les vrais chiffres de la base avant de calculer quoi que ce soit (volume, couverture, données manquantes) et vérifier nos filtres sur des palmarès connus.

Prérequis : `bash scripts/download_data.sh` depuis la racine du dépôt.

**Notez les chiffres obtenus dans `report/donnees.md`** : ils servent directement au rapport.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # pour importer src/ depuis notebooks/

import pandas as pd
import matplotlib.pyplot as plt
from src.load import load_matches, load_players, load_rankings, LEVELS

matches = load_matches()
players = load_players()
rankings = load_rankings()
print(f"{len(matches):,} matchs, saisons {matches.year.min()}–{matches.year.max()}")
print(f"{len(players):,} joueurs, {len(rankings):,} lignes de classement")

## 1. Volume par saison

La dernière saison est-elle complète ? Y a-t-il des trous ?

In [ ]:
per_year = matches.groupby("year").size()
per_year.plot(kind="bar", figsize=(14, 3), title="Matchs du circuit principal par saison")
plt.show()
per_year.tail()

## 2. Niveaux de tournoi et surfaces

Vérifier les codes de `tourney_level` avec `data/raw/tennis_atp/matches_data_dictionary.txt`.

In [ ]:
print(matches.tourney_level.value_counts())
print(matches.surface.value_counts(dropna=False))
print(matches["round"].value_counts())

## 3. Couverture des statistiques de match

À partir de quand les stats de service sont-elles disponibles ? (attendu : vers 1991)

In [ ]:
cover = matches.assign(has_stats=matches.w_svpt.notna()).groupby("year").has_stats.mean()
cover.plot(figsize=(14, 3), title="Part des matchs avec statistiques", ylim=(0, 1))
plt.show()
cover.loc[1988:1995]

## 4. Valeurs manquantes par colonne

In [ ]:
matches.isna().mean().sort_values(ascending=False).round(3).head(20)

## 5. Couverture des classements

Attendu : quasi complet depuis 1985, partiel 1973–1984, 1982 absent.

In [ ]:
rk_per_year = rankings.groupby(rankings.ranking_date.dt.year).ranking_date.nunique()
rk_per_year.plot(kind="bar", figsize=(14, 3), title="Nombre de classements publiés par an")
plt.show()

## 6. Contrôles sur des palmarès connus

Comparer aux palmarès officiels (site de l'ATP, Wikipédia). Un écart signale un problème de filtre ou de données.

In [ ]:
gs_finals = matches[(matches.tourney_level == "G") & (matches["round"] == "F")]
gs_finals.winner_name.value_counts().head(15)

In [ ]:
from src.indicators import weeks_at_no1
w1 = weeks_at_no1(rankings).merge(players[["player_id", "name"]], on="player_id")
w1.head(15)

## 7. Sélection des joueurs

Critère provisoire : au moins 3 Grands Chelems **ou** au moins une semaine au n°1. Combien de joueurs ? Le critère est-il juste pour les années 70 ?

In [ ]:
slams = gs_finals.winner_id.value_counts()
selected = set(slams[slams >= 3].index) | set(w1[w1.weeks_no1 > 0].player_id)
print(len(selected), "joueurs")
players[players.player_id.isin(selected)][["player_id", "name", "ioc", "dob"]].sort_values("dob")